# CyberGuard - **Voice: real-world upgrade** (Kaggle) - fewer false alarms on real voices

On **In-the-Wild** (real recordings of 58 celebrities and politicians from the internet) your AI-voice model has a
good AUC (0.967) but wrongly flags **23.8% of the REAL recordings** at its alert level: its small classifier and its
levels were made on clean studio speech only. This notebook fixes exactly that part.

* `voice_ai.onnx` (the big speech model) is **not changed** - its features are used as they are.
* The small classifier on top and the alert / check-carefully levels (all inside `voice_ai.json`) are trained again on
  the same data as before (ASVspoof 2019, MLAAD English, LibriSpeech) **plus half of the In-the-Wild celebrities**.
* The levels are set so that only 1% (alert) / 5% (check carefully) of **each** kind of real voice reaches them -
  studio speech, ASVspoof speakers **and** real-world recordings.
* **Honest test:** the other In-the-Wild celebrities (their voices were never heard in training or for choosing
  anything), ASVspoof 2019 eval, MLAAD held-out systems + LibriSpeech test-clean, and In-the-Wild after WhatsApp-style
  Opus compression. Your **installed** classifier is scored on exactly the same recordings.

Output: one file, **`voice_ai.json`**. On your PC, `install_voice_head.py` installs it only if it is better.

**Before you start (Kaggle)**
1. **Add Input -> Your Work / Datasets ->** your private dataset with `voice_ai.onnx` + `voice_ai.json`
   (e.g. `cyberguard-voice-model` from the In-the-Wild test).
2. Right panel -> **Session options: Accelerator GPU T4 x2, Internet On**.
3. **Add-ons -> Secrets: `HF_TOKEN`** (your Hugging Face *Read* token; MLAAD needs it) - tick it for this notebook.
4. **Save Version -> Save & Run All (Commit)**. About 3-5 hours (downloads ~1-2 h, features ~1.5-2 h). You can close
   the browser. Then download `voice_ai.json` from the version's **Output** tab.

## 1. Settings

In [ ]:
import os, sys, glob, json, time, random, shutil, subprocess, hashlib, io, csv, posixpath, zipfile
from collections import Counter
import numpy as np
SMOKE = os.environ.get('SMOKE') == '1'          # only for testing this notebook on a tiny fake dataset
KAGGLE = os.path.isdir('/kaggle/working')
def _roomiest(paths):                                         # the 8 GB zip goes where there is most space
    ok = []
    for p in paths:
        try:
            os.makedirs(p, exist_ok=True); ok.append((shutil.disk_usage(p).free, p))
        except OSError:
            pass
    return max(ok)[1]
WORK = os.environ.get('CG_WORK') or (_roomiest(['/kaggle/temp/cg', '/tmp/cg']) if KAGGLE else '/content/cg')  # temporary
OUT = os.environ.get('CG_OUT') or ('/kaggle/working' if KAGGLE else '/content/cg_out')   # the result files
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
REPO, ZIP_NAME = 'mueller91/In-The-Wild', 'release_in_the_wild.zip'
OPUS_N = 20 if SMOKE else 3000          # recordings for the WhatsApp-style test
MAX_MEMBER = 200_000_000                # no single recording in the zip may be bigger than this (200 MB)
GPU_BATCH = int(os.environ.get('CG_GPU_BATCH', '16'))   # 4-second clips per GPU batch (about 2-3 GB of GPU memory)
SEED = 42
GPU = (not SMOKE) and shutil.which('nvidia-smi') is not None and \
      subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode == 0
print('SMOKE TEST' if SMOKE else 'REAL RUN', '| Kaggle' if KAGGLE else '| Colab/other', '| GPU' if GPU else '| no GPU (CPU)',
      '| work:', WORK, '| free disk:', round(shutil.disk_usage(WORK).free / 1e9), 'GB')

## 2. Tools (fixed versions)

In [ ]:
if not SMOKE:
    pip = [sys.executable, '-m', 'pip', '-q']
    pyav = 'av==19.0.2' if sys.version_info >= (3, 12) else 'av==17.0.0'     # 19.x needs Python 3.12+ (same results)
    subprocess.run(pip + ['install', pyav, 'huggingface_hub', 'scikit-learn', 'pandas'], check=True)   # no upgrades
    ort_gpu = False
    if GPU:                     # onnxruntime-gpu 1.23.2 = CUDA 12 (Kaggle's PyTorch brings the CUDA 12 libraries)
        subprocess.run(pip + ['uninstall', '-y', 'onnxruntime'], check=False)
        ort_gpu = subprocess.run(pip + ['install', 'onnxruntime-gpu==1.23.2']).returncode == 0
    if not ort_gpu:
        subprocess.run(pip + ['install', 'onnxruntime==1.23.2'], check=True)
import av, sklearn, onnxruntime as ort
if 'CUDAExecutionProvider' in ort.get_available_providers():
    try:
        import torch                   # loads the CUDA / cuDNN libraries onnxruntime-gpu needs
    except Exception:
        try:
            ort.preload_dlls()
        except Exception as e:
            print('CUDA libraries not preloaded:', type(e).__name__)
print('PyAV', av.__version__, '| onnxruntime', ort.__version__, ort.get_available_providers(), '| scikit-learn', sklearn.__version__)

## 3. The audio code (identical to CyberGuard's `voice_audio.py`)

In [ ]:
VOICE_AUDIO = r'''"""Audio in, clean 16 kHz speech out - ONE implementation shared by the Colab training notebook
(docs/colab/CyberGuard_Voice_Colab.ipynb copies this file word for word) and the scanner on the PC, so the voice
model sees exactly the same kind of audio in training and in real use.

  any voice note / recording (WhatsApp .opus/.ogg, .m4a, .mp3, .aac, .wav, .flac, .webm ...)
     --PyAV (bundled FFmpeg; the file is only decoded, never run)--> mono, 16 kHz, float32
     --trim silence at both ends (a known shortcut: in ASVspoof 2019 real clips have longer silences than fakes)
     --cut into 4-second clips (short audio is repeated up to 4 s) --> each clip normalised (zero mean, unit variance,
       exactly what the XLS-R speech model expects)

Also: codec round-trips (training augmentation: phone-quality compression applied to real AND fake clips alike, so
compression can never be a shortcut) and file facts for the forensics checks (format, codec, sample rate, tags).
Only NumPy + PyAV.
"""
from __future__ import annotations

import io
import math

import numpy as np

SR = 16_000
CLIP_S = 4.0
CLIP = int(SR * CLIP_S)
MIN_SPEECH_S = 1.0
MAX_SECONDS = 600            # at most 10 minutes of a recording are decoded
MAX_CLIPS = 15               # at most 15 clips (1 minute of speech, spread over the recording) are scored


def _av():
    import av                # imported here so the module loads (and its tests run) where PyAV is missing
    return av


def decode(src, sr: int | None = SR, max_seconds: float = MAX_SECONDS) -> tuple[np.ndarray, int]:
    """(mono float32 samples in [-1, 1], sample rate). src: a path or bytes. sr=None keeps the file's own rate.
    Raises ValueError for anything that is not decodable audio."""
    av = _av()
    try:
        container = av.open(io.BytesIO(src) if isinstance(src, (bytes, bytearray)) else src)
    except (av.error.FFmpegError, OSError, ValueError) as exc:          # type: ignore[attr-defined]
        raise ValueError(f"not a readable audio file ({type(exc).__name__})") from None
    try:
        streams = [s for s in container.streams if s.type == "audio"]
        if not streams:
            raise ValueError("the file has no audio track")
        stream = streams[0]
        rate = sr or stream.codec_context.sample_rate or SR
        resampler = av.AudioResampler(format="flt", layout="mono", rate=rate)
        chunks, n, limit = [], 0, int(rate * max_seconds)
        try:
            for frame in container.decode(stream):
                for out in resampler.resample(frame):
                    a = out.to_ndarray().reshape(-1)
                    chunks.append(a)
                    n += a.size
                if n >= limit:
                    break
            for out in resampler.resample(None):                          # flush
                chunks.append(out.to_ndarray().reshape(-1))
        except (av.error.FFmpegError, ValueError):                        # type: ignore[attr-defined]
            if not chunks:
                raise ValueError("the audio could not be decoded") from None   # cut-off file: keep what was read
    finally:
        container.close()
    x = np.concatenate(chunks)[:limit].astype(np.float32) if chunks else np.zeros(0, np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(x, -1.0, 1.0), rate


def file_facts(src) -> dict:
    """Format / codec / sample rate / channels / duration / bit rate / metadata tags (no audio decoded)."""
    av = _av()
    try:
        c = av.open(io.BytesIO(src) if isinstance(src, (bytes, bytearray)) else src)
    except (av.error.FFmpegError, OSError, ValueError):                   # type: ignore[attr-defined]
        return {}
    try:
        s = next((s for s in c.streams if s.type == "audio"), None)
        tags = {str(k).lower(): str(v)[:200] for k, v in dict(c.metadata or {}).items()}
        if s is not None:
            tags.update({str(k).lower(): str(v)[:200] for k, v in dict(s.metadata or {}).items()})
        return {"container": c.format.name if c.format else "", "codec": s.codec_context.name if s else "",
                "sampleRate": int(s.codec_context.sample_rate or 0) if s else 0,
                "channels": int(s.codec_context.channels or 0) if s else 0,
                "seconds": round(float(c.duration) / 1_000_000, 2) if c.duration else None,
                "bitRate": int(c.bit_rate or 0), "tags": tags}
    finally:
        c.close()


def trim_silence(x: np.ndarray, sr: int = SR, floor_db: float = -45.0, rel_db: float = -35.0) -> np.ndarray:
    """Cut silence at the start and end (20 ms frames quieter than both an absolute floor and 35 dB under the
    loudest frame). Silence in the middle is kept - pauses are part of speech."""
    hop = max(1, int(sr * 0.02))
    n = x.size // hop
    if n == 0:
        return x
    frames = x[: n * hop].reshape(n, hop)
    db = 10 * np.log10(np.mean(frames.astype(np.float64) ** 2, axis=1) + 1e-12)
    voiced = np.where(db > max(floor_db, db.max() + rel_db))[0]
    if voiced.size == 0:
        return x[:0]
    return x[voiced[0] * hop: (voiced[-1] + 1) * hop]


def normalize(clip: np.ndarray) -> np.ndarray:
    """Zero mean, unit variance - the input normalisation the XLS-R / wav2vec2 models were trained with."""
    c = clip.astype(np.float32)
    return ((c - c.mean()) / np.sqrt(c.var() + 1e-7)).astype(np.float32)


def clips(x: np.ndarray, max_clips: int = MAX_CLIPS) -> list[np.ndarray]:
    """Speech -> normalised 4-second clips. Short speech (1-4 s) is repeated up to 4 s; under 1 s -> no clips.
    Long speech: clips spread evenly over the whole recording (not just the start)."""
    if x.size < int(SR * MIN_SPEECH_S):
        return []
    if x.size < CLIP:
        x = np.tile(x, math.ceil(CLIP / x.size))[:CLIP]
    n = x.size // CLIP
    starts = [i * CLIP for i in range(n)]
    if x.size - n * CLIP >= SR * MIN_SPEECH_S:            # the tail is worth a clip of its own (aligned to the end)
        starts.append(x.size - CLIP)
    if len(starts) > max_clips:
        idx = np.linspace(0, len(starts) - 1, max_clips).round().astype(int)
        starts = [starts[i] for i in idx]
    return [normalize(x[s: s + CLIP]) for s in starts]


def speech(src) -> np.ndarray:
    """File -> 16 kHz mono speech with the silent ends trimmed."""
    x, _ = decode(src)
    return trim_silence(x)


# ---- training augmentation (used by the notebook; harmless on the PC) ------------------------------------------
_CODECS = {"opus": ("ogg", "libopus", 48_000), "mp3": ("mp3", "libmp3lame", 16_000), "aac": ("adts", "aac", 16_000)}


def codec_roundtrip(x: np.ndarray, codec: str = "opus", bitrate: int = 16_000) -> np.ndarray:
    """Encode 16 kHz audio with a phone / messenger codec and decode it again (in memory)."""
    av = _av()
    fmt, enc, rate = _CODECS[codec]
    buf = io.BytesIO()
    out = av.open(buf, mode="w", format=fmt)
    st = out.add_stream(enc, rate=rate)
    st.bit_rate = bitrate
    st.layout = "mono"
    res = av.AudioResampler(format=st.format.name if st.format else "s16", layout="mono", rate=rate)
    frame = av.AudioFrame.from_ndarray(x.astype(np.float32).reshape(1, -1), format="flt", layout="mono")
    frame.sample_rate = SR
    for f in res.resample(frame) + res.resample(None):
        for p in st.encode(f):
            out.mux(p)
    for p in st.encode(None):
        out.mux(p)
    out.close()
    y, _ = decode(buf.getvalue())
    return y[: x.size] if y.size >= x.size else np.pad(y, (0, x.size - y.size))


def telephone(x: np.ndarray) -> np.ndarray:
    """Narrow-band phone line: down to 8 kHz and back (keeps only 300-3400 Hz-ish speech band)."""
    import numpy.fft as F
    spec = F.rfft(x)
    freqs = F.rfftfreq(x.size, 1 / SR)
    spec[(freqs < 300) | (freqs > 3400)] = 0
    return F.irfft(spec, n=x.size).astype(np.float32)
'''

with open(os.path.join(WORK, 'voice_audio.py'), 'w', encoding='utf-8') as f:
    f.write(VOICE_AUDIO)
sys.path.insert(0, WORK)
import importlib, voice_audio as V
importlib.reload(V)
print('voice_audio ready: clips of', V.CLIP_S, 's at', V.SR, 'Hz, at most', V.MAX_CLIPS, 'clips per recording')

## 4. Find and check the model files
`voice_ai.json` stores the SHA-256 of the `voice_ai.onnx` it was made with; both must match (the PC checks the same).

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''):
            h.update(b)
    return h.hexdigest()

def find_model():
    if os.environ.get('CG_MODEL_DIR'):
        cands = [os.path.join(os.environ['CG_MODEL_DIR'], 'voice_ai.onnx')]
    else:
        cands = (glob.glob('/kaggle/input/**/voice_ai.onnx', recursive=True)
                 + glob.glob('/content/voice_ai.onnx') + glob.glob('/content/*/voice_ai.onnx')
                 + glob.glob('/content/*/*/voice_ai.onnx'))
    pairs, problems = {}, []
    for o in cands:
        j = os.path.join(os.path.dirname(o), 'voice_ai.json')
        if not (os.path.isfile(o) and os.path.isfile(j)):
            problems.append(f'{o}: voice_ai.json is not next to it'); continue
        with open(j, encoding='utf-8') as f:
            m = json.load(f)
        s = sha256_file(o)
        if m.get('onnx_sha256') != s:
            problems.append(f'{o}: does not belong to the voice_ai.json next to it'); continue
        pairs.setdefault(s, (o, m))
    for p in problems:
        print('skipped:', p)
    if not pairs:
        raise RuntimeError('voice_ai.onnx + voice_ai.json not found. Add them as Input (see the top of this notebook).')
    if len(pairs) > 1:
        raise RuntimeError('Different trained models were found - keep only one as Input: '
                           + ', '.join(p[0] for p in pairs.values()))
    s, (o, m) = next(iter(pairs.items()))
    return o, m, s

ONNX_PATH, META, MODEL_SHA = find_model()
if META.get('smoke_test') and not SMOKE:
    raise RuntimeError('This voice_ai.json comes from a smoke test, not from the real training run.')
LEVELS = META['levels']
ALERT, VERIFY = float(LEVELS['alert']), float(LEVELS['verify'])
MEAN, SCALE = np.asarray(META['scaler_mean']), np.asarray(META['scaler_scale'])
COEF, INTERCEPT = np.asarray(META['coef']), META['intercept']
print('model:', ONNX_PATH, '| SHA-256', MODEL_SHA[:16] + '...', '| trained', META.get('trained_at'))
print(f'alert level {ALERT:.4f} | check-carefully level {VERIFY:.4f} | layer {META.get("layer")}')
print('results already in voice_ai.json:', list(META.get('results', {})))

def clip_probs(feats):
    """the classifier on top of the speech features - the same formula as ai_voice_score() in app/detectors/voice.py"""
    z = (feats - MEAN) / SCALE
    return 1 / (1 + np.exp(-(z @ COEF + INTERCEPT)))

CPU_SESS = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
GPU_SESS = None
if 'CUDAExecutionProvider' in ort.get_available_providers():
    try:
        g = ort.InferenceSession(ONNX_PATH, providers=[('CUDAExecutionProvider', {'use_tf32': 0}), 'CPUExecutionProvider'])
        if g.get_providers()[0] == 'CUDAExecutionProvider':
            GPU_SESS = g
    except Exception as e:
        print('GPU session not available:', type(e).__name__, str(e)[:200])
print('GPU session ready (checked against the CPU below)' if GPU_SESS else 'running on the CPU, like the PC')

## 5. Download In-the-Wild and check it
The exact file published on Hugging Face (its SHA-256 is compared after the download). About 8.2 GB; on Kaggle this
takes a few minutes.

In [ ]:
def hf_token():
    """Kaggle Secrets, Colab Secrets or the HF_TOKEN environment variable (never printed)."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')

if SMOKE:
    ZIP = os.environ['CG_ITW_ZIP']
    DATASET = {'repo': 'SMOKE TEST (fake zip)', 'file': os.path.basename(ZIP), 'sha256': sha256_file(ZIP)}
else:
    from huggingface_hub import HfApi, hf_hub_download
    tok = hf_token() or None
    try:
        api = HfApi()
        REV = api.dataset_info(REPO, token=tok).sha
        pi = api.get_paths_info(REPO, [ZIP_NAME], repo_type='dataset', revision=REV, token=tok)[0]
        lfs = pi.lfs
        WANT_SHA = lfs.sha256 if hasattr(lfs, 'sha256') else lfs['sha256']
        WANT_SIZE = int(pi.size)
        need = WANT_SIZE + 1_000_000_000
        if shutil.disk_usage(WORK).free < need and not os.path.exists(os.path.join(WORK, 'itw', ZIP_NAME)):
            raise RuntimeError(f'not enough disk space in {WORK}: {need / 1e9:.1f} GB needed')
        t0 = time.time()
        ZIP = hf_hub_download(REPO, ZIP_NAME, repo_type='dataset', revision=REV, token=tok,
                              local_dir=os.path.join(WORK, 'itw'))
    except Exception as e:
        print('Download failed:', type(e).__name__, str(e)[:300])
        print('If it mentions 401 / 403 / gated: open https://huggingface.co/datasets/mueller91/In-The-Wild, log in, '
              'accept the terms if asked, add your HF_TOKEN secret (Add-ons -> Secrets), then run again.')
        raise
    size = os.path.getsize(ZIP)
    print(f'downloaded {size / 1e9:.2f} GB in {time.time() - t0:.0f}s - checking SHA-256 ...')
    got = sha256_file(ZIP)
    if size != WANT_SIZE or got != WANT_SHA:
        os.remove(ZIP)
        raise RuntimeError('The download is not the published file (size or SHA-256 differs) - it was deleted; run again.')
    print('SHA-256 matches the published file')
    DATASET = {'repo': REPO, 'file': ZIP_NAME, 'revision': REV, 'sha256': got, 'bytes': size,
               'licence': 'CC BY-SA 4.0 (Hugging Face dataset card)'}

## 6. Read the labels (straight from the zip, nothing unpacked)
`meta.csv` lists every recording with its speaker and label (`bona-fide` = real, `spoof` = deepfake).

In [ ]:
ZF = zipfile.ZipFile(ZIP)
MEMBERS = {i.filename: i for i in ZF.infolist()
           if not i.is_dir() and '__MACOSX' not in i.filename and not posixpath.basename(i.filename).startswith('._')}

def itw_label(v):
    v = str(v).strip().lower()
    if v in ('spoof', 'fake', 'deepfake', '1'):
        return 1
    if v in ('bona-fide', 'bonafide', 'bona fide', 'real', 'genuine', '0'):
        return 0
    return None

ROWS, META_CSV, UNKNOWN, MISSING = [], None, Counter(), 0
for m in [n for n in MEMBERS if posixpath.basename(n).lower() == 'meta.csv']:
    rd = csv.DictReader(io.StringIO(ZF.read(MEMBERS[m]).decode('utf-8-sig')))
    cols = {c.strip().lower(): c for c in (rd.fieldnames or [])}
    fcol = cols.get('file') or cols.get('path') or cols.get('filename')
    lcol, scol = cols.get('label'), cols.get('speaker')
    if not (fcol and lcol):
        continue
    base = posixpath.dirname(m)
    rows, unknown, missing, seen = [], Counter(), 0, set()
    for r in rd:
        lab = itw_label(r[lcol])
        if lab is None:
            unknown[str(r[lcol])] += 1; continue
        name = posixpath.normpath(posixpath.join(base, str(r[fcol]).strip()))
        if name not in MEMBERS:
            missing += 1; continue
        if name in seen:
            continue
        seen.add(name)
        rows.append({'file': name, 'label': lab, 'speaker': str(r[scol]).strip() if scol else '?'})
    if len(rows) > len(ROWS):
        ROWS, META_CSV, UNKNOWN, MISSING = rows, m, unknown, missing
if not ROWS:                                                       # no meta.csv: labels as folder names (fake/ real/)
    for n in sorted(MEMBERS):
        if n.lower().endswith(('.wav', '.flac', '.mp3')):
            lab = next((itw_label(q) for q in reversed(n.lower().split('/')[:-1]) if itw_label(q) is not None), None)
            if lab is not None:
                ROWS.append({'file': n, 'label': lab, 'speaker': '?'})
    META_CSV = 'folder names'
if not ROWS:
    raise RuntimeError('No labelled recordings found. Files in the zip: ' + ', '.join(sorted(MEMBERS)[:15]))
print(f'labels from {META_CSV}: {sum(1 - r["label"] for r in ROWS):,} real + {sum(r["label"] for r in ROWS):,} deepfake '
      f'recordings, {len({r["speaker"] for r in ROWS})} speakers')
if UNKNOWN or MISSING:
    print(f'not used: {sum(UNKNOWN.values())} rows with an unknown label {dict(UNKNOWN.most_common(5))}, '
          f'{MISSING} rows whose file is not in the zip')

## 7. The training notebook's data (the same sources the installed classifier was made with)

## 4. Data
Every download is checked; anything optional that fails is reported and skipped, never silently.

In [ ]:
def run(cmd):
    print('$', ' '.join(cmd) if isinstance(cmd, list) else cmd)
    subprocess.run(cmd, check=True, shell=isinstance(cmd, str))

def fetch(urls, dest, min_bytes):
    """Download with resume (wget -c), trying each mirror; checks the size."""
    if os.path.exists(dest) and os.path.getsize(dest) >= min_bytes:
        return dest
    for u in urls:
        try:
            run(['wget', '-q', '-c', '--tries=3', '-O', dest + '.part', u])
            if os.path.getsize(dest + '.part') >= min_bytes:
                os.replace(dest + '.part', dest)
                return dest
            print('  too small, trying the next mirror')
        except subprocess.CalledProcessError:
            print('  failed, trying the next mirror')
        if os.path.exists(dest + '.part') and os.path.getsize(dest + '.part') < 1e6:
            os.remove(dest + '.part')
    raise RuntimeError(f'Could not download {os.path.basename(dest)} - run this cell again')

def hf_token():
    """Kaggle Secrets, Colab Secrets or the HF_TOKEN environment variable (never printed)."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')

### 4a. ASVspoof 2019 LA (required, 7.1 GB)

In [ ]:
def smoke_audio(path, fake, seed, seconds=None):
    """SMOKE only: a fake 'voice' (harmonics + noise) so the pipeline can run without real data."""
    import soundfile as sf
    r = np.random.default_rng(seed)
    sec = seconds or r.uniform(2.0, 6.0)
    t = np.arange(int(V.SR * sec)) / V.SR
    f0 = r.uniform(90, 220)
    s = sum(np.sin(2 * np.pi * f0 * k * t + r.uniform(0, 6)) / k for k in range(1, 8))
    s *= 0.5 + 0.5 * np.sin(2 * np.pi * r.uniform(2, 5) * t) ** 2
    s += r.normal(0, 0.02 if fake else 0.2, t.size)
    sil = np.zeros(int(V.SR * r.uniform(0.1, 0.6)))
    sf.write(path, (0.2 * np.concatenate([sil, s, sil]) / np.abs(s).max()).astype(np.float32), V.SR)

LA = os.path.join(WORK, 'LA')
if SMOKE and not os.path.isdir(LA):
    import soundfile as sf
    os.makedirs(os.path.join(LA, 'ASVspoof2019_LA_cm_protocols'))
    for split, sysids, tag in (('train', ['A01', 'A02'], 'trn'), ('dev', ['A01', 'A02'], 'trl'), ('eval', ['A07', 'A16'], 'trl')):
        d = os.path.join(LA, f'ASVspoof2019_LA_{split}', 'flac'); os.makedirs(d)
        lines = []
        for i in range(40):
            fake = i % 2 == 1
            name = f'LA_{split[0].upper()}_{i:07d}'
            smoke_audio(os.path.join(d, name + '.flac'), fake, hash((split, i)) % 10000)
            lines.append(f'LA_{i % 5:04d} {name} - {sysids[i % 2] if fake else "-"} {"spoof" if fake else "bonafide"}')
        open(os.path.join(LA, 'ASVspoof2019_LA_cm_protocols', f'ASVspoof2019.LA.cm.{split}.{tag}.txt'), 'w').write('\n'.join(lines) + '\n')
elif not os.path.isdir(os.path.join(LA, 'ASVspoof2019_LA_train')):
    z = fetch(['https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip',
               'https://datashare.is.ed.ac.uk/bitstream/handle/10283/3336/LA.zip'], os.path.join(WORK, 'LA.zip'), 7.0e9)
    run(['unzip', '-q', '-o', z, '-d', WORK])
    os.remove(z)                                                     # free 7 GB

def la_protocol(split):
    tag = 'trn' if split == 'train' else 'trl'
    rows = []
    for line in open(os.path.join(LA, 'ASVspoof2019_LA_cm_protocols', f'ASVspoof2019.LA.cm.{split}.{tag}.txt')):
        p = line.split()
        if len(p) == 5:
            path = os.path.join(LA, f'ASVspoof2019_LA_{split}', 'flac', p[1] + '.flac')
            rows.append({'path': path, 'label': int(p[4] == 'spoof'), 'system': p[3] if p[3] != '-' else 'bonafide',
                         'group': 'asvspoof19', 'speaker': p[0]})
    assert rows and all(os.path.exists(r['path']) for r in rows[:50]), f'ASVspoof {split} files missing'
    return rows

LA_ROWS = {s: la_protocol(s) for s in ('train', 'dev', 'eval')}
for s, rows in LA_ROWS.items():
    print(f'ASVspoof 2019 LA {s}: {sum(1 - r["label"] for r in rows):,} real, {sum(r["label"] for r in rows):,} fake, '
          f'systems {sorted({r["system"] for r in rows} - {"bonafide"})}')

### 4b. LibriSpeech real speech (train-clean-100 + dev-clean for training/validation, test-clean for tests and voiceprint)

In [ ]:
def librispeech(part):
    root = os.path.join(WORK, 'LibriSpeech', part)
    if SMOKE and not os.path.isdir(root):
        for spk in range(8):
            d = os.path.join(root, str(100 + spk), '1'); os.makedirs(d)
            for u in range(6):
                smoke_audio(os.path.join(d, f'{100 + spk}-1-{u:04d}.flac'), False, 1000 + spk * 10 + u)
    elif not os.path.isdir(root):
        z = fetch([f'https://www.openslr.org/resources/12/{part}.tar.gz', f'https://us.openslr.org/resources/12/{part}.tar.gz',
                   f'https://openslr.elda.org/resources/12/{part}.tar.gz'], os.path.join(WORK, part + '.tar.gz'), 3.0e8)
        run(['tar', '-xzf', z, '-C', WORK]); os.remove(z)
    rows = [{'path': p, 'label': 0, 'system': 'bonafide', 'group': 'librispeech', 'speaker': p.split(os.sep)[-3]}
            for p in sorted(glob.glob(os.path.join(root, '*', '*', '*.flac')))]
    assert rows, f'LibriSpeech {part} has no audio'
    print(f'LibriSpeech {part}: {len(rows):,} recordings, {len({r["speaker"] for r in rows})} speakers')
    return rows

LS_DEV, LS_TEST = librispeech('dev-clean'), librispeech('test-clean')
LS_TRAIN = librispeech('train-clean-100')      # 251 more real speakers for training (6.3 GB) - balances the many fakes

### 4c. MLAAD - modern TTS / voice-cloning systems (optional; needs `HF_TOKEN` and the dataset's *Agree* click)
Only the folders needed are listed (never the whole dataset): the English systems, then up to `PER_MODEL` clips of
each. English systems are split **by system** into train / validation / test, so the test systems were never seen.
Downloads run 64 at a time; if they take longer than `MLAAD_MAX_MINUTES`, training continues with what arrived.

In [ ]:
MLAAD = []
PER_MODEL = 6 if SMOKE else 140          # 143 English systems x 140 = about 20,000 recordings
MLAAD_MAX_MINUTES = 120
if not SMOKE:
    tok = hf_token()
    if not tok:
        print('No HF_TOKEN secret - MLAAD skipped (the model is then trained on ASVspoof 2019 only).')
    else:
        try:
            from huggingface_hub import HfApi, hf_hub_download
            from huggingface_hub.hf_api import RepoFolder
            from concurrent.futures import ThreadPoolExecutor, as_completed
            api = HfApi(token=tok)
            REPO = 'mueller91/MLAAD'
            def ls(path):
                """One folder only (not the whole dataset): -> (folders, audio files)."""
                items = list(api.list_repo_tree(REPO, path_in_repo=path or None, repo_type='dataset', recursive=False))
                return ([i.path for i in items if isinstance(i, RepoFolder)],
                        [i.path for i in items if not isinstance(i, RepoFolder)
                         and i.path.lower().endswith(('.wav', '.flac', '.mp3'))])
            t0 = time.time()
            root_dirs, _ = ls('')
            fake = next((d for d in root_dirs if d.rstrip('/').split('/')[-1] == 'fake'), None)
            if fake is None:                                       # e.g. a version folder first: look one level down
                for d in root_dirs:
                    sub, _ = ls(d)
                    fake = next((s for s in sub if s.rstrip('/').split('/')[-1] == 'fake'), None)
                    if fake:
                        break
            assert fake, f'no "fake" folder found; top level: {root_dirs[:10]}'
            langs_found, _ = ls(fake)
            en = next((l for l in langs_found if l.rstrip('/').split('/')[-1] == 'en'), None)
            assert en, f'no English folder; languages: {[l.split("/")[-1] for l in langs_found]}'
            systems, _ = ls(en)
            print(f'MLAAD: {len(systems)} English systems found under {en} ({time.time() - t0:.0f}s)', flush=True)
            rng = random.Random(SEED)
            pick = []
            for s in sorted(systems):
                sub, files = ls(s)
                if not files and sub:                              # audio one folder deeper
                    files = [f for d in sub[:3] for f in ls(d)[1]]
                files = sorted(files); rng.shuffle(files)
                pick += [(s.rstrip('/').split('/')[-1], f) for f in files[:PER_MODEL]]
            print(f'MLAAD: downloading {len(pick):,} files from {len(systems)} systems ({time.time() - t0:.0f}s) ...', flush=True)
            local = os.path.join(WORK, 'mlaad')
            def _get(sf):
                p = os.path.join(local, sf[1])
                if os.path.exists(p) and os.path.getsize(p) > 0:   # already here (an earlier run)
                    return p
                try:
                    return hf_hub_download(REPO, sf[1], repo_type='dataset', token=tok, local_dir=local)
                except Exception:
                    return None
            got, t1, stopped = {}, time.time(), False
            ex = ThreadPoolExecutor(64)
            futs = {ex.submit(_get, sf): sf for sf in pick}
            try:
                for k, fu in enumerate(as_completed(futs), 1):
                    if not fu.cancelled():                             # a cancelled (never started) download is skipped
                        got[futs[fu]] = fu.result()
                    if k % 1000 == 0:
                        print(f'  MLAAD: {k:,} / {len(pick):,} files ({(time.time() - t1) / 60:.0f} min)', flush=True)
                    if not stopped and time.time() - t1 > MLAAD_MAX_MINUTES * 60:
                        stopped = True
                        for f2 in futs:
                            f2.cancel()                                 # queued downloads are dropped, running ones finish
                        print(f'  MLAAD: time limit reached - continuing with the files already downloaded', flush=True)
            finally:
                ex.shutdown(wait=True, cancel_futures=True)     # no download keeps running after this cell
            for (system, f) in pick:
                p = got.get((system, f))
                if p and os.path.exists(p):
                    MLAAD.append({'path': p, 'label': 1, 'system': system, 'group': 'mlaad', 'lang': 'en', 'speaker': system})
            print(f'MLAAD: {len(MLAAD):,} of {len(pick):,} files downloaded ({(time.time() - t0) / 60:.0f} min)')
        except Exception as e:                                     # optional source: never stop the run
            print('MLAAD skipped:', type(e).__name__, str(e)[:300])
            MLAAD = []
else:
    for m in [f'tts_model_{i}' for i in range(7)]:
        d = os.path.join(WORK, 'mlaad', 'fake', 'en', m); os.makedirs(d, exist_ok=True)
        for u in range(PER_MODEL):
            p = os.path.join(d, f'{m}_{u}.wav'); smoke_audio(p, True, hash((m, u)) % 10000)
            MLAAD.append({'path': p, 'label': 1, 'system': m, 'group': 'mlaad', 'lang': 'en', 'speaker': m})
print(f'MLAAD used: {len(MLAAD):,} fake recordings, {len({r["system"] for r in MLAAD})} English systems')

## 8. Splits
* **ASVspoof / MLAAD / LibriSpeech:** exactly the same rules as the training notebook (MLAAD systems split by system,
  LibriSpeech by speaker).
* **In-the-Wild: split by celebrity** (each person's real AND deepfake recordings stay together): about 50% of the
  people -> training, 15% -> validation (chooses the classifier and the levels), 35% -> **test** (never used for anything).

In [ ]:
import hashlib
rng = random.Random(SEED)
def sample(rows, n):
    rows = list(rows); rng.shuffle(rows); return rows[:n]
def per_system(rows, n_each):
    out = []
    for s in sorted({r['system'] for r in rows}):
        out += sample([r for r in rows if r['system'] == s], n_each)
    return out
def md5(s):
    return hashlib.md5(s.encode()).hexdigest()

N_TRAIN_FAKE_PER_SYS = 10 if SMOKE else 1300
la_tr, la_dv, la_ev = LA_ROWS['train'], LA_ROWS['dev'], LA_ROWS['eval']
en_systems = sorted({r['system'] for r in MLAAD if r['lang'] == 'en'})
sys_split = {s: ('test' if i % 7 == 0 else 'val' if i % 7 == 1 else 'train') for i, s in enumerate(sorted(en_systems, key=md5))}
ls_spk = sorted({r['speaker'] for r in LS_DEV}, key=md5)
ls_val_spk = set(ls_spk[: max(1, len(ls_spk) // 4)])

ITW_ROWS = [dict(r, group='itw', system='itw') for r in ROWS]
itw_spk = sorted({r['speaker'] for r in ITW_ROWS}, key=md5)
assert SMOKE or len(itw_spk) >= 20, f'In-the-Wild speakers not found ({len(itw_spk)}) - the split by person needs them'
ITW_PART = {s: ('train' if i < 0.50 * len(itw_spk) else 'val' if i < 0.65 * len(itw_spk) else 'test')
            for i, s in enumerate(itw_spk)}
ITW_SPLIT = {p: [r for r in ITW_ROWS if ITW_PART[r['speaker']] == p] for p in ('train', 'val', 'test')}

SPLITS = {
    'train': [r for r in la_tr if r['label'] == 0] + per_system([r for r in la_tr if r['label'] == 1], N_TRAIN_FAKE_PER_SYS)
             + [r for r in MLAAD if r['lang'] == 'en' and sys_split[r['system']] == 'train']
             + [r for r in LS_DEV if r['speaker'] not in ls_val_spk] + sample(LS_TRAIN, 40 if SMOKE else 8000)
             + ITW_SPLIT['train'],
    'val': [r for r in la_dv if r['label'] == 0] + per_system([r for r in la_dv if r['label'] == 1], 10 if SMOKE else 500)
           + [r for r in MLAAD if r['lang'] == 'en' and sys_split[r['system']] == 'val']
           + [r for r in LS_DEV if r['speaker'] in ls_val_spk] + ITW_SPLIT['val'],
}
ITW_KEY, OPUS_KEY = 'In-the-Wild (real-world, outside test)', 'In-the-Wild after WhatsApp-style Opus 16 kbps'
TESTS = {
    'ASVspoof 2019 eval (11 of 13 attacks unseen)': sample([r for r in la_ev if r['label'] == 0], 40 if SMOKE else 3000)
        + per_system([r for r in la_ev if r['label'] == 1], 10 if SMOKE else 450),
    'MLAAD held-out English systems + LibriSpeech test-clean real': [r for r in MLAAD if r['lang'] == 'en' and sys_split[r['system']] == 'test']
        + sample(LS_TEST, 40 if SMOKE else 1500),
    ITW_KEY: ITW_SPLIT['test'],
}
OPUS_ROWS = sample(ITW_SPLIT['test'], 20 if SMOKE else 3000)
for k, v in list(SPLITS.items()) + list(TESTS.items()) + [(OPUS_KEY, OPUS_ROWS)]:
    print(f'{k:62s} {sum(1 - r["label"] for r in v):6,} real {sum(r["label"] for r in v):6,} fake')
print('In-the-Wild people ->', {p: sum(1 for v in ITW_PART.values() if v == p) for p in ('train', 'val', 'test')})

## 9. Recordings -> 4-second clips -> speech features (with YOUR installed `voice_ai.onnx`)
Training recordings: one random 4-second clip each, passed through a random codec (none / Opus 8-24 kbps / MP3 /
AAC / phone band) - the same as the training notebook, for real and fake alike. Validation and test recordings:
exactly what the PC does (up to 15 clips spread over the recording). The GPU is first checked to give the same
features as the CPU (the PC runs on the CPU).

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def augment(x, r):
    u = r.random()
    if u < 0.40:
        return x
    try:
        if u < 0.65:
            return V.codec_roundtrip(x, 'opus', int(r.choice([8000, 12000, 16000, 24000])))
        if u < 0.80:
            return V.codec_roundtrip(x, 'mp3', 32000)
        if u < 0.92:
            return V.codec_roundtrip(x, 'aac', 32000)
        return V.telephone(x)
    except Exception:
        return x

def speech_of(r):
    if r.get('group') == 'itw':
        info = MEMBERS[r['file']]
        if info.file_size > MAX_MEMBER:
            raise ValueError('too big')
        return V.speech(ZF.read(info))             # the zip checks each file's CRC while reading
    return V.speech(r['path'])

def prepare(job):
    i, r, mode, seed = job
    try:
        x = speech_of(r)
    except Exception:
        return i, []
    if mode == 'train':
        g = random.Random(seed)
        if x.size > V.CLIP:
            s = g.randrange(0, x.size - V.CLIP + 1); x = x[s: s + V.CLIP]
        x = augment(x.astype(np.float32), g)
        return i, V.clips(V.trim_silence(x) if x.size else x, max_clips=1)
    if mode == 'opus16':
        x = V.codec_roundtrip(x, 'opus', 16000)
    return i, V.clips(x)

def run_feats(sess, X, batch):
    return np.concatenate([sess.run(None, {'wav': X[k:k + batch]})[0] for k in range(0, len(X), batch)])

SESS, BATCH = CPU_SESS, 4
if GPU_SESS is not None:
    g = random.Random(SEED)
    probe = [c for r in g.sample(SPLITS['val'], min(12, len(SPLITS['val']))) for c in prepare((0, r, 'eval', 0))[1]][:24]
    if probe:
        X = np.stack(probe).astype(np.float32)
        a, b = clip_probs(run_feats(CPU_SESS, X, 4)), clip_probs(run_feats(GPU_SESS, X, GPU_BATCH))
        diff = float(np.max(np.abs(a - b)))
        print(f'GPU vs CPU on {len(X)} clips: largest score difference {diff:.6f}')
        if diff < 0.002:
            SESS, BATCH = GPU_SESS, GPU_BATCH
print('features with', SESS.get_providers()[0])

THREADS = max(2, min(16, (os.cpu_count() or 2) * 2))
def featurize(rows, mode, name):
    """-> (features float32 (clips, dim), clip -> recording index, labels per recording, group per recording)"""
    t0 = time.time()
    feats, owner, buf, bown = [], [], [], []
    def flush():
        if buf:
            feats.append(run_feats(SESS, np.stack(buf).astype(np.float32), BATCH)); owner.extend(bown); buf.clear(); bown.clear()
    jobs = [(i, r, mode, SEED * 100003 + i) for i, r in enumerate(rows)]
    with ThreadPoolExecutor(THREADS) as ex:
        for k in range(0, len(jobs), 256):
            for i, cl in ex.map(prepare, jobs[k:k + 256]):
                for c in cl:
                    buf.append(c); bown.append(i)
                    if len(buf) >= 4 * BATCH:
                        flush()
            if k and k % 4096 == 0:
                el = time.time() - t0
                print(f'  {name}: {k:,} / {len(jobs):,} recordings ({el / 60:.0f} min, about {el / k * (len(jobs) - k) / 60:.0f} min left)', flush=True)
    flush()
    F = np.concatenate(feats).astype(np.float32) if feats else np.zeros((0, len(MEAN)), np.float32)
    print(f'{name}: {len(rows):,} recordings -> {len(owner):,} clips ({len(rows) - len(set(owner))} unreadable / too short) '
          f'in {(time.time() - t0) / 60:.1f} min', flush=True)
    return F, np.asarray(owner, int), np.asarray([r['label'] for r in rows]), np.asarray([r.get('group', '?') for r in rows])

DATA = {'train': featurize(SPLITS['train'], 'train', 'train'), 'val': featurize(SPLITS['val'], 'eval', 'validation')}
for name, rows in TESTS.items():
    DATA[name] = featurize(rows, 'eval', name)
DATA[OPUS_KEY] = featurize(OPUS_ROWS, 'opus16', OPUS_KEY)

## 10. Train the classifier again (chosen on validation only)
Logistic regression on the same speech features. Tried: strength C x how much the In-the-Wild training voices count.
Chosen by the **average EER of the three kinds of validation voice** (ASVspoof, studio TTS + LibriSpeech, real-world),
so no kind is sacrificed for another. Levels: the highest of the three kinds' 99% (alert) / 95% (verify) real scores.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve

def eer(y, s):
    fpr, tpr, _ = roc_curve(y, s); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return float((fpr[i] + fnr[i]) / 2)

def recording_scores(clip_p, owner, n):
    s = np.full(n, np.nan); sums = np.zeros(n); cnt = np.zeros(n)
    np.add.at(sums, owner, clip_p); np.add.at(cnt, owner, 1)
    ok = cnt > 0; s[ok] = sums[ok] / cnt[ok]; return s

KIND = {'asvspoof19': 'ASVspoof', 'mlaad': 'studio', 'librispeech': 'studio', 'itw': 'real-world'}
Ftr, Otr, Ytr, Gtr = DATA['train']; Fva, Ova, Yva, Gva = DATA['val']
ytr, gtr = Ytr[Otr], Gtr[Otr]
kva = np.asarray([KIND.get(g, g) for g in Gva])

def head_scores(sc, lr, name):
    F, O, Y, G = DATA[name]
    return recording_scores(lr.predict_proba(sc.transform(F))[:, 1], O, len(Y))

best = None
print('    C  ITW weight   ASVspoof EER  studio EER  real-world EER   mean')
for C in (0.001, 0.01, 0.1):
    sc = StandardScaler().fit(Ftr)
    Xtr = sc.transform(Ftr)
    for w in (1.0, 3.0, 10.0):
        sw = np.where(gtr == 'itw', w, 1.0)
        lr = LogisticRegression(C=C, max_iter=3000, class_weight='balanced').fit(Xtr, ytr, sample_weight=sw)
        s = head_scores(sc, lr, 'val')
        e = {}
        for k in ('ASVspoof', 'studio', 'real-world'):
            m = (kva == k) & ~np.isnan(s)
            if len(set(Yva[m])) == 2:
                e[k] = eer(Yva[m], s[m])
        mean = float(np.mean(list(e.values())))
        print(f'{C:6.3f} {w:6.0f}        ' + '   '.join(f'{e.get(k, float("nan")):10.4f}' for k in ('ASVspoof', 'studio', 'real-world'))
              + f'   {mean:.4f}')
        if best is None or mean < best[0] - 1e-9:
            best = (mean, C, w, sc, lr, e)
VAL_MEAN_EER, C_BEST, W_ITW, SCALER, CLF, VAL_EERS = best
print(f'\nChosen: C={C_BEST}, In-the-Wild weight {W_ITW} (validation mean EER {VAL_MEAN_EER:.2%})')

s_val = head_scores(SCALER, CLF, 'val')
LEVEL_BY_KIND = {}
for k in ('ASVspoof', 'studio', 'real-world'):
    real = s_val[(kva == k) & (Yva == 0) & ~np.isnan(s_val)]
    if len(real):
        LEVEL_BY_KIND[k] = (float(np.quantile(real, 0.99)), float(np.quantile(real, 0.95)), int(len(real)))
NEW_ALERT = max(v[0] for v in LEVEL_BY_KIND.values())
NEW_VERIFY = min(max(v[1] for v in LEVEL_BY_KIND.values()), NEW_ALERT)
print('real validation voices per kind (99% / 95% score):', {k: (round(a, 4), round(b, 4), n) for k, (a, b, n) in LEVEL_BY_KIND.items()})
print(f'new alert level {NEW_ALERT:.4f} | check-carefully level {NEW_VERIFY:.4f}   (installed: {ALERT:.4f} / {VERIFY:.4f})')

## 11. Honest tests - new classifier vs your INSTALLED one, on exactly the same recordings
Each at its own levels. **AUC** (1.0 = perfect ranking), **EER**, deepfakes caught and real recordings wrongly flagged.

In [ ]:
def summarize(s, y, alert, verify):
    ok = ~np.isnan(s); s, y = s[ok], y[ok]
    r = {'recordings': int(len(y)), 'real': int((y == 0).sum()), 'fake': int((y == 1).sum())}
    if (y == 1).any():
        r['fake caught at alert'] = round(float((s[y == 1] >= alert).mean()), 4)
        r['fake caught at verify'] = round(float((s[y == 1] >= verify).mean()), 4)
    if (y == 0).any():
        r['real flagged at alert (false alarms)'] = round(float((s[y == 0] >= alert).mean()), 4)
        r['real flagged at verify'] = round(float((s[y == 0] >= verify).mean()), 4)
    if (y == 0).any() and (y == 1).any():
        r['AUC'] = round(float(roc_auc_score(y, s)), 4); r['EER'] = round(eer(y, s), 4)
    r['not scored (unreadable or under 1 s of speech)'] = int((~ok).sum())
    return r

def installed_scores(name):
    F, O, Y, G = DATA[name]
    return recording_scores(clip_probs(F), O, len(Y))

NEW, OLD = {}, {}
for name in ['val'] + list(TESTS) + [OPUS_KEY]:
    Y = DATA[name][2]
    NEW[name] = summarize(head_scores(SCALER, CLF, name), Y, NEW_ALERT, NEW_VERIFY)
    OLD[name] = summarize(installed_scores(name), Y, ALERT, VERIFY)
for k in (ITW_KEY, OPUS_KEY):
    spk = {r['speaker'] for r in (TESTS[k] if k in TESTS else OPUS_ROWS)}
    NEW[k]['speakers'] = OLD[k]['speakers'] = len(spk)
    NEW[k]['note'] = OLD[k]['note'] = ('celebrities never used for training or for choosing anything (the other '
                                       'In-the-Wild celebrities were used for training)')
for name in ['val'] + list(TESTS) + [OPUS_KEY]:
    print(f'\n{name}')
    for m in ('AUC', 'EER', 'fake caught at alert', 'real flagged at alert (false alarms)', 'fake caught at verify', 'real flagged at verify'):
        if m in NEW[name]:
            print(f'   {m:38s} installed {OLD[name].get(m)!s:>8}   new {NEW[name].get(m)!s:>8}')

## 12. The new `voice_ai.json`
Everything about the speech model stays as it is (same `onnx_sha256`); only the classifier, the levels and the results
change. The installed classifier's results on the same recordings are stored in it, so the PC can compare fairly.

In [ ]:
import datetime, copy
new_meta = copy.deepcopy(META)
new_meta.update({
    'scaler_mean': SCALER.mean_.tolist(), 'scaler_scale': SCALER.scale_.tolist(),
    'coef': CLF.coef_[0].tolist(), 'intercept': float(CLF.intercept_[0]), 'C': C_BEST,
    'levels': {'alert': NEW_ALERT, 'verify': NEW_VERIFY,
               'rule': 'alert: 1% of real validation recordings of EACH kind (ASVspoof, studio, real-world) reach it; verify: 5%',
               'by_kind': {k: {'alert_99': a, 'verify_95': b, 'real_recordings': n} for k, (a, b, n) in LEVEL_BY_KIND.items()}},
    'validation_eer': VAL_MEAN_EER, 'validation_eer_by_kind': VAL_EERS,
    'results': NEW,
    'head_version': 2,
    'head_upgrade': {'what': 'classifier + levels trained again with half of the In-the-Wild celebrities; voice_ai.onnx unchanged',
                     'itw_weight': W_ITW,
                     'itw_people': {p: sum(1 for v in ITW_PART.values() if v == p) for p in ('train', 'val', 'test')},
                     'itw_dataset': DATASET},
    'installed_head_same_test': {'trained_at': META.get('trained_at'), 'levels': {'alert': ALERT, 'verify': VERIFY}, 'results': OLD},
    'smoke_test': bool(SMOKE),
    'trained_at': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
})
td = dict(META.get('training_data') or {})
td['In-the-Wild celebrities (training half, real + deepfake recordings)'] = len(ITW_SPLIT['train'])
new_meta['training_data'] = td
new_meta['versions'] = dict(META.get('versions') or {}, head_upgrade={'onnxruntime': ort.__version__, 'scikit-learn': sklearn.__version__})
assert new_meta['onnx_sha256'] == MODEL_SHA
with open(os.path.join(OUT, 'voice_ai.json'), 'w', encoding='utf-8') as f:
    json.dump(new_meta, f, indent=1)
print('wrote', os.path.join(OUT, 'voice_ai.json'))
print('\nDONE - download voice_ai.json from the Output tab (voice_ai.onnx is unchanged - do not download it again).')